<h1 style="color:black; font-family:Arial, sans-serif;">🏡 Scraping Immobilier – Avito.ma</h1>

<hr style="border:1px solid #ccc;">

<h2 style="color:#117A65;">Data Transformation</h2>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
import mysql.connector

In [ ]:
import warnings

warnings.filterwarnings('ignore')


In [ ]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database="avito_db"
)

cursor = conn.cursor()

In [ ]:
query = "SELECT * FROM Avito_maison"
df = pd.read_sql(query, conn)
df.head()

In [ ]:
df.shape[0]

In [ ]:
df.columns

In [ ]:
# Replace 'Nan' strings with actual NaN values
df = df.replace('Nan', np.nan)

In [ ]:
df = df[~df['Surface_Totale'].isna()]
df = df[~df['Prix'].isna()]
print(df.shape[0])

In [ ]:
df['Ville'].unique()

In [ ]:
counts = df["Ville"].value_counts()

villes_a_garder = counts[counts >20].index
print("✅ Villes gardées :", list(villes_a_garder))

df = df[df["Ville"].isin(villes_a_garder)].copy()


In [ ]:

df['Ville'].value_counts() 

In [ ]:
# 1. Remapping des petites villes vers des zones personnalisées
remap_villes = {
    "Mohammedia": "Alentours de Casablanca",
    "Bouskoura": "Alentours de Casablanca",
    "Berrechid": "Alentours de Casablanca",
    "Settat": "Alentours de Casablanca",
    "Bouznika": "Alentours de Casablanca",
    "Nouaceur": "Alentours de Casablanca",
    "Had Soualem": "Alentours de Casablanca",
    "Deroua": "Alentours de Casablanca",
    "Temara": "Alentours de Rabat",
    "Skhirat": "Alentours de Rabat",
    "Khemisset": "Alentours de Rabat",
    "Sidi Slimane": "Alentours de Rabat",
    "Sidi Allal El Bahraoui": "Alentours de Rabat",
    "Ain Aouda": "Alentours de Rabat",
    "Tifelt": "Alentours de Rabat",
    "Tétouan": "Nord",
    "Larache": "Nord",
    "Martil": "Nord",
    "Asilah": "Nord",
    "Al Hoceima": "Nord",
    "Ksar el-Kebir": "Nord",
    "Azemmour": "Centre",
    "Safi": "Marrakech-Safi",
    "Nador": "Oriental",
    "Berkane": "Oriental",
    "Dakhla": "Sud",
    "Laâyoune": "Sud",
    "Tiznit": "Sud",
    "Ouarzazate": "Sud",
    "Taroudant": "Sud",
    "Midelt": "Oriental",
    "Guercif": "Oriental",
    "Azilal": "Centre",
    "Ben Guerir": "Marrakech-Safi",
    "Sidi Rahal": "Marrakech-Safi",
    "Khouribga": "Centre",
    "Sidi Kacem": "Centre",
    "Béni Mellal": "Centre",
    "Khénifra": "Centre",
    "El Kelâa des Sraghna": "Marrakech-Safi",
}

# 2. Appliquer ce remapping (ne modifie que les villes listées)
df["Ville"] = df["Ville"].replace(remap_villes)

print(df["Ville"].value_counts())

In [ ]:
# Calculate the number of nulls in each column
nulls_per_column = df.isnull().sum()

# Calculate the total number of nulls in the DataFrame
total_nulls = nulls_per_column.sum()

# Calculate the total number of unique records that contain at least one null value
total_unique_records_with_nulls = len(df[df.isnull().any(axis=1)].drop_duplicates())

print("Nulls in each feature:")
print(nulls_per_column)
print("\nTotal nulls in the DataFrame:", total_nulls)
print("\nTotal records with at least one null value: ", total_unique_records_with_nulls)

In [ ]:
df['Surface_Totale'] = pd.to_numeric(df['Surface_Totale'], errors='coerce')
df['Surface_Habitable'] = pd.to_numeric(df['Surface_Habitable'], errors='coerce')
df["Surface_Habitable"] = df["Surface_Habitable"].astype(float)
df["Surface_Totale"] = df["Surface_Totale"].astype(float)
df["Surface_Habitable"] = df["Surface_Habitable"].fillna(df["Surface_Totale"] * 0.95)

In [ ]:
df.drop(columns=["Etage"], inplace=True)

In [ ]:
df['Condition_encoded'] = df['Condition'].map({'Neuf':0, 'Bon état':1, 'à rénover':2})
df['Standing_encoded'] = df['Standing'].map({'Economique':0, 'Moyen standing':1, 'Haut standing':2})
df['Age_encoded'] = df['Age'].map({'Moins de 1 an': 0, '1-5 ans': 1, '6-10 ans': 2, '11-20 ans': 3, '21+ ans': 4})

df.head()

In [ ]:
df['Chambres'].value_counts()

In [ ]:
# Convertit en numérique, remplace les erreurs (comme 'Nan') par NaN
df['Chambres'] = pd.to_numeric(df['Chambres'], errors='coerce')

df['Chambres'] = df['Chambres'].apply(lambda x: '7' if x > 6 else x)


In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
# Convertit en numérique, remplace les erreurs (comme 'Nan') par NaN
df['Salles_De_Bain'] = pd.to_numeric(df['Salles_De_Bain'], errors='coerce')

df['Salles_De_Bain'] = df['Salles_De_Bain'].apply(lambda x: '5' if x > 4 else x)


In [ ]:
df['Salons'].value_counts()

In [ ]:
# Convertit en numérique, remplace les erreurs (comme 'Nan') par NaN
df['Salons'] = pd.to_numeric(df['Salons'], errors='coerce')

df['Salons'] = df['Salons'].apply(lambda x: '5' if x > 4 else x)


In [ ]:
# Calculate the number of nulls in each column
nulls_per_column = df.isnull().sum()

# Calculate the total number of nulls in the DataFrame
total_nulls = nulls_per_column.sum()

# Calculate the total number of unique records that contain at least one null value
total_unique_records_with_nulls = len(df[df.isnull().any(axis=1)].drop_duplicates())

print("Nulls in each feature:")
print(nulls_per_column)
print("\nTotal nulls in the DataFrame:", total_nulls)
print("\nTotal records with at least one null value: ", total_unique_records_with_nulls)

In [ ]:
# Function to convert a single price to Dirhams
def convert_to_dirhams(price):
    if isinstance(price, str):
        price = price.strip()
        if price == '' or price.lower() in ['nan', 'none']:
            return None
        price = price.replace(',', '.').replace('\xa0', '').replace('DH', '').replace(' ', '')
        try:
            return float(price)
        except ValueError:
            return None
    return price


# Apply the function to each value in the 'Prix' column
df['Prix'] = df['Prix'].apply(convert_to_dirhams)

# Convert the non-null values in the 'Prix' column to integer
df.loc[df['Prix'].notna(), 'Prix'] = df.loc[df['Prix'].notna(), 'Prix'].astype(int)

df.head()

In [ ]:
df.columns

In [ ]:
columns = ['Terrasse', 'Parking', 'Securite', 'Balcon', 'Chauffage',
       'Climatisation', 'Cuisine_equipee', 'Meuble', 'Garage', 'Jardin',
       'Piscine']
for column in columns:
    df[column] = df[column].replace({"oui": 1, "Non": 0})
    
df.head()

In [ ]:
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy='median')
df['Salles_De_Bain'] = num_imputer.fit_transform(df[['Salles_De_Bain']])
df['Chambres'] = num_imputer.fit_transform(df[['Chambres']])
df['Salons'] = num_imputer.fit_transform(df[['Salons']])

In [ ]:
from rapidfuzz import fuzz, process

# Exemple : liste unique de quartiers
quartiers = df['Quartier'].unique()

# Dictionnaire de correspondance
corrections = {}

# Boucle pour comparer chaque quartier aux autres
for i, q1 in enumerate(quartiers):
    for q2 in quartiers[i+1:]:
        score = fuzz.ratio(q1.lower(), q2.lower())
        if score >= 85:  # seuil de similarité
            print(f"{q1} <--> {q2} : {score}% similaire")
            # On choisit la forme la plus courte comme forme standard
            standard = min(q1, q2, key=len)
            corrections[q1] = standard
            corrections[q2] = standard

print("\nDictionnaire de remplacement :")
print(corrections)


In [ ]:
corrections = {'Centre ville': 'Centre Ville', 'Ancienne medina': 'Ancienne Médina', 'Charf': 'Charaf', 
               'Massira 1': 'Massira', 'Massira 2': 'Massira', 'Massira 3': 'Massira', 
               'Marjane 2': 'Marjane', 'Hay Al Falah': 'Hay Alfalah', 'Anahda': 'Nahda',
               'El Bassatine': 'Al Bassatine', 'Hay massira': 'Hay Massira', 'Hay Essalam': 'Hay Salam', 
               'Hay essalam': 'Hay Salam', 'Hay salam': 'Hay Salam', 'Hay Al Hassani': 'Hay Hassani', 
               'Hassania 1': 'Hassania', 'Hassania 2': 'Hassania', 'Zone Industrielle': 'Zone industrielle', 
               'Biranzarane': 'Bir Anzarane', 'Branes 2': 'Branes', 'Branes 1': 'Branes', 
               'Marchane': 'Marchan', 'Beausejour': 'Beauséjour',
               'Sidi AbdelKrim': 'Sidi AbdelKarim', 'Hay Moulay Rachid 2': 'Hay Moulay Rachid', 
               'Hay Moulay Rachid 6': 'Hay Moulay Rachid', 'Hay Moulay Rachid 3': 'Hay Moulay Rachid', 
               'Nahda-2': 'Nahda', 'Nahda-1': 'Nahda', 'Bir Rami Est': 'Bir Rami', 'Bir Rami Ouest': 'Bir Rami', 
               'Bd Allal el fassi': 'Allal El Fassi', 'Hay saada': 'Hay Saada',
               'Val Fleuri': 'Val Fleury', 'Hay Riyad': 'Hay Riad', 'Hay El Farah':'Hay Al Farah'}
# Fonction de remplacement
def corriger_quartier(q):
    return corrections.get(q, q)  # si pas dans le dict, garder original

df['Quartier'] = df['Quartier'].apply(corriger_quartier)

In [ ]:
# Calculer la moyenne du prix par quartier
quartier_mean = df.groupby('Quartier')['Prix'].mean()

# Associer chaque ligne à la moyenne de son quartier
df['Quartier_enc'] = df['Quartier'].map(quartier_mean)

In [ ]:

df['Age_encoded'] = df.groupby('Quartier_enc')['Age_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Age_encoded'] = df['Age_encoded'].fillna(df['Age_encoded'].median())
df['Age_encoded'] = df['Age_encoded'].round().astype(int)


df['Condition_encoded'] = df.groupby('Quartier_enc')['Condition_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Condition_encoded'] = df['Condition_encoded'].fillna(df['Condition_encoded'].median())
df['Condition_encoded'] = df['Condition_encoded'].round().astype(int)

df['Standing_encoded'] = df.groupby('Quartier_enc')['Standing_encoded'].transform(
    lambda x: x.fillna(x.median())
)
df['Standing_encoded'] = df['Standing_encoded'].fillna(df['Standing_encoded'].median())
df['Standing_encoded'] = df['Standing_encoded'].round().astype(int)

df['Standing_encoded'].unique()


In [ ]:
df = df[df['Prix']>50000] 

In [ ]:
df['Prix'].describe()

<h2 style="color:black"> Using IQR </h2>

In [ ]:
df["Salles_De_Bain"] = pd.to_numeric(df["Salles_De_Bain"], errors="coerce")
Q1 = df["Salles_De_Bain"].quantile(0.25)
Q3 = df["Salles_De_Bain"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition0 = (df["Salles_De_Bain"] < (Q1 - 1.5 * IQR)) | (df["Salles_De_Bain"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliers = df[outlier_condition0]["Salles_De_Bain"]

# Print the number of outliers
print("\nNumber of outliers in Nombre De Salles_De_Bain:", outliers.count())

In [ ]:
df['Salles_De_Bain'].value_counts()

In [ ]:
df["Chambres"] = pd.to_numeric(df["Chambres"], errors="coerce")
Q1 = df["Chambres"].quantile(0.25)
Q3 = df["Chambres"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition1 = (df["Chambres"] < (Q1 - 1.5 * IQR)) | (df["Chambres"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliers = df[outlier_condition1]["Chambres"]

# Print the number of outliers
print("\nNumber of outliers in Nombre De Chambres:", outliers.count())

In [ ]:
df["Salons"] = pd.to_numeric(df["Salons"], errors="coerce")
Q1 = df["Salons"].quantile(0.25)
Q3 = df["Salons"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition2 = (df["Salons"] < (Q1 - 1.5 * IQR)) | (df["Salons"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliers = df[outlier_condition2]["Salons"]

# Print the number of outliers
print("\nNumber of outliers in Nombre De Salons:", outliers.count())

In [ ]:
Q1 = df["Prix"].quantile(0.25)
Q3 = df["Prix"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition = (df["Prix"] < (Q1 - 1.5 * IQR)) | (df["Prix"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQP = df[outlier_condition]

# Print the number of outliers
print("\nNumber of outliers in Prix:", outliersQP.shape[0])

In [ ]:
Q1 = df["Surface_Totale"].quantile(0.25)
Q3 = df["Surface_Totale"].quantile(0.75)
IQR = Q3 - Q1
# Define a condition for an outlier
outlier_condition3 = (df["Surface_Totale"] < (Q1 - 1.5 * IQR)) | (df["Surface_Totale"] > (Q3 + 1.5 * IQR))
# Get the outliers
outliersQS = df[outlier_condition3]
# Print the number of outliers
print("\nNumber of outliers in Surface:", outliersQS.shape[0])

In [ ]:
plt.figure(figsize=(4, 4.6))
# Melt the DataFrame to create a suitable format for the boxplot
df_melted = df.melt(value_vars=["Chambres", "Salles_De_Bain", "Salons"], var_name="Variable", value_name="Value")

sns.boxplot(x="Variable", y="Value", data=df_melted, hue="Variable", palette=['purple', 'orange', 'red'], width=0.6, legend=False)
plt.ylabel('Values')
plt.title('Comparison of Interquartile Ranges')

# Reduce the size of the x-axis labels
plt.xticks(fontsize=6)

plt.show()

In [ ]:
plt.figure(figsize=(4, 5))

# Subplot 1: Surface
plt.subplot(1, 2, 1)
sns.boxplot(y=df["Surface_Totale"], color='green')
plt.title('IQR of Surface')
plt.ylim(0, 800)  # Set the y-axis limits from 0 to 1000

# Subplot 2:'Prix'
plt.subplot(1, 2, 2)
sns.boxplot(y=df["Prix"],width=0.6, color='orange')
plt.title('IQR of Prix')
plt.ylim(0, 9000000)  # Set the y-axis limits from 0 to 1000

plt.tight_layout()
plt.show()

<h2 style="color:black"> Using z-score </h2>

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6)) 
plot = sns.histplot(df['Surface_Totale'], binwidth=15, color='orange', ax=ax)
ax.set_xlabel('Surface')
ax.set_ylabel('Frequency')
ax.set_xlim(0, 2000)
ax.set_aspect('equal')  
plt.show()

In [ ]:
plt.figure(figsize=(6.6, 4.6))

plt.scatter( df['Surface_Totale'] ,df['Prix'] ,color='#0080ff', label='Training Data')
plt.xlabel("Surface")
plt.ylabel('Prix')
plt.xlim(0, 2000)
plt.ylim(0, 9000000)
plt.show()

In [ ]:
z_scores = np.abs(stats.zscore(df['Prix']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers
print(df['Prix'][outliersZ].count())

In [ ]:
z_scores = np.abs(stats.zscore(df['Surface_Totale']))
threshold = 3
outliersZ = z_scores > threshold

# Print the outliers number
print(df['Prix'][outliersZ].count())

In [ ]:
outliersQP.describe()

In [ ]:
df.shape[0]
df = df[~outlier_condition]
df.shape[0]

In [ ]:
outliersQS.describe()

In [ ]:
#  a condition for the outliers to keep
keep_condition = (outliersQS["Surface_Habitable"] < 2000.0) 
# Get the outliers to keep
outliers_to_keep = outliersQS[keep_condition]

# Get the outliers to remove
outliers_to_remove = outliersQS[~keep_condition]

# Remove the outliers from the DataFrame
df = df.drop(outliers_to_remove.index)

In [ ]:
df.shape[0]

In [ ]:
import numpy as np

# Remplacer les vrais NaN (np.nan) ou chaînes "nan"/"NaN" par None
df = df.replace(["nan", "NaN", np.nan], None)

# Liste des colonnes sans 'id'
columns = [col for col in df.columns if col != "id"]

# Construction de la requête
cols_str = ", ".join(f"`{col}`" for col in columns)
placeholders = ", ".join(["%s"] * len(columns))
insert_query = f"INSERT INTO maison_Transformed ({cols_str}) VALUES ({placeholders})"

# Insertion dans MySQL
for _, row in df[columns].iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()


In [ ]:
cursor.close()
conn.close()
